# Decorators

Decorators are a way to add bolt on functionality to an existing
function, they allow coders to make more ‘pythonic’ code by wrapping (or
decorating) functions rather than having to write longer functions or
manually code repetitive functionality in multiple places and allow each
function you write to stand alone and focus on doing one thing well.

In this tutorial we will cover:

- What decorators are,
- Generalising decorators,
- Example use cases for decorators
  - **Measuring runtime** - check how long a function takes to run.
  - **Creating logs to aid debugging** log runtime information to a
    separate file to allow coders and users to review situational data.
  - **Parameterization of reporting** create reports for different
    areas, organisations or customers.
- Potential issues when working with decorators

> **Download**
>
> You can download the Jupyter Notebook with the completed examples by
> clicking the “Jupyter” link under “Other Formats”. If following Code
> Club Live you may wish to download at the start of the session.

## What Are Decorators?

Simply put, a decorator is a standalone function that takes another
function or as it’s input and performs operations around it without
modifying the original function. It allows a programmer to add
functionality to existing code without manually combining the code from
the 2 functions and that doesn’t change the function.

Decorators take advantage of the fact that Python treats functions as
first-order objects that can be directly referred to just like simpler
objects such as strings, lists and tuples. That is to say it is an
object we can assign to a variable or pass to or return from another
function.

In [1]:
# define a simple function
def stuff():
    return 'doing stuff'

# assign function to a variable        
work = stuff

# print the variable 'work' to demonstrate it is an object
print(work)

<function stuff at 0x0000015C3F177E20>

> **Note**
>
> When we print the variable **hi** we do not see the message “hello”,
> this is because we haven’t yet called the function. To call the
> function we must insert closed brackets **()** after the function
> name. If trying to assign a function rather than its result you **must
> not** call it when assigning.

In [2]:
# print the called function 'work'
print(work())

doing stuff

When we build a decorator we are building a closure, that is to say we
are packaging one function inside another that when the outer function
is called it creates a lingering representation of the inner function
with the arguments and objects it was passed when the outer function was
executed and that can be called at further into the process. this allows
us to create multiple instances of the decorator around different
functions or with different arguments. Put more simply a decorator can
be applied to multiple other functions and more than one decorator can
be applied to a given function.

The following example demonstrates a simple decorator factory with an
inner and outer function.

In [3]:
# This is the outer function which closes the inner wrapper around another function
def outer_function(func): # we could use any variable name but 'func' is the wrapped variable.
    # This is the function that will be wrapped around our function
    def inner_wrapper(): 
        # This code runs before the function
        print('Waking up')
        # This is when you want the function to run
        result = func()
        # This code runs after the function
        print('Going to sleep')
        # Now we return 'result'
        return result
    # this is where we return our decorated function
    return inner_wrapper

Having created our decorator we now need to apply it to our function we
can use the familiar syntax by assigning using ‘=’ as we did to make
‘stuff’ into ‘work’:

In [4]:
# we can assign a wrapper to a function 
stuff = outer_function(stuff)

print(stuff())

Waking up
Going to sleep
doing stuff

Essentially we are taking advantage of the fact that we can re-assign
the wrapped function to a variable with the name ‘stuff’ - the same name
as our original function. Once we remember this we can use shorthand
notation using @ to assign a wrapped function to its own name (though
this needs to be applied to the definition of the function):

In [5]:
# this is how we assign a decorator to a function using '@' notation
@outer_function
def more_stuff():
    return 'Looks like there was more to do'

print(more_stuff())

Waking up
Going to sleep
Looks like there was more to do

It is worth noting that whilst decorating a function does not alter the
decorated functions code it can lead to some interesting predicaments
which we will discuss a little further down.

### Dealing With Arguments

So far the examples have been fairly simple and not included any
arguments, something that is very common. Note how the below code
generates an error:

In [6]:
# @outer_function
# def drinks(drink, milk, sugar):
#     return f"I'd love a {drink} with {milk} and {sugar} sugars"

# print(drinks('tea', 'plain', sugar=2))

The reason this fails is to do with the way our decorator is built, to
handle the arguments we need to make sure our wrapper layer knows to
expect them. We can do this by passing each argument individually to the
wrapper, but this leaves us with a decorator that can only handle those
specific variables and that will still break if we have a function with
different arguments. The way we get round that in python is by using the
notation “\*args, \*\*kwargs”, this essentially represents any number of
positional arguments and key word arguments (if you would like to know
more [Visually Explained](https://www.youtube.com/watch?v=FFpDsC6B2qw)
does a really good job of explaining this).

Lets build a new decorator that can handle this:

In [7]:
def drink_decorator(func):
    # note the placement of *args, **kwargs in both the wrapper and then when we call the function.
    def wrapper(*args, **kwargs):
        # This code runs before the function
        print("I went to the cafe")
        # This is when you want the function to run
        result = func(*args, **kwargs)
        # This code runs after the function
        print("I decided what I want")
        # Now we return 
        return result
    # this is where we return our decorated function
    return wrapper

@drink_decorator
def drinks(drink, milk, sugar):
    return f"I'd love a {drink} with {milk} and {sugar} sugars"

print(drinks('tea', 'milk', sugar=2))
print(drinks('coffee', 'cream', 'no'))

I went to the cafe
I decided what I want
I'd love a tea with milk and 2 sugars
I went to the cafe
I decided what I want
I'd love a coffee with cream and no sugars

You can see this now runs.

## Generalising Decorators

So far we’ve used simple decorators that have had specific use cases but
lets try decorating our ‘stuff’ function with our ‘drink_decorator’

In [8]:
@drink_decorator
def stuff():
    return 'doing stuff'

print(stuff())

I went to the cafe
I decided what I want
doing stuff

The real power of a decorator comes when we generalise it so we can use
its functionality in many different situations. Decorators are at their
most powerful when they can be applied to a broad scope of functions and
you may find that once you’ve built a decorator in one project it has
scope for use in others.

To achieve this we can bear the following in mind:

- Always allow for differing arguments using “\*args, \*\*kwargs”.
- Make the name of your outer function as generic as you can whilst
  keeping it as meaningful as possible (if you are looking to time the
  execution of a function ‘func_timer’ is perfectly acceptable.)
- Use general names and terms within your function, sticking with
  convention where possible (for example the inner function is normally
  just called ‘wrapper’).
- Parameterise your decorators where it make sense to (we’ll cover this
  in the third example below).

When we move on to looking at our examples we wil hopefully see these
principles at work.

## Real World Examples

In this section we are going to work through three real work related
examples that are common use cases for decorators. Each of these is
going to help us explore some of the potential upsides and issues that
we might find with decorators.

### Measuring Time

A common problem in coding is understanding how long elements of code
take to run. It is possible to add markers to a function or script to
time the execution, but decorators offer us a way to wrap this
functionality around existing code.

In this example we are going to simulate a function with a longer
execution time using time.sleep (which causes Python to wait at
execution for a given number of seconds when called). We will also be
using the time module to record start and stop times in our decorator.

In [9]:
import time
from functools import wraps # This line will be explained in example 2

def func_timer(func):
    @wraps(func) # This line will be explained in example 2
    def wrapper(*args, **kwargs):
        start = time.time() # assign the start time to a variable
        result = func(*args, **kwargs)
        end = time.time() # assign the end time to a variable
        # This will print after the decorated function has run but before the result is returned
        print(f"[TIME] {func.__name__} took {end - start:.4f} seconds") # print the time taken.
        # func.__name__ gives the name of the decorated function.
        return result
    return wrapper


# decorated function
@func_timer
def slow_function(run_time):
    print("I'll get on that")
    time.sleep(run_time)
    print("I've done that")
    return run_time

slow_function(7)

I'll get on that
I've done that
[TIME] slow_function took 7.0031 seconds

7

This is a generic decorator that uses the `__name__` method of the
decorated function to report back the time it took to run. This allows a
more meaningful response to the user than simply reporting the runtime.
you could take this decorator, copy it into your code and apply the tag
`@func_timer` to any of your functions and it would report back in the
same way. This is the key benefit of decorators, portability and
generalisation.

There are alterations that we could make here, but we will demonstrate
some further examples and then allow you to imagine how you might add
further functionality to improve generalisation.

### Generating Logs

Debugging and resolving errors as well as looking for efficiencies and
how to speed up slow corner cases are all common scenarios for analysts
to find themselves in. We know that a problem occurred but we don’t
always understand what caused an error or slow run-time. The decorator
we are going to build here is a useful way of logging situational data
such as the name of the function that ran, as well as the arguments that
were used and potentially more.

In this example we are going to create a log file into which the data is
recorded the file is created when the decorated function is first
defined and then updated each time that function is called.

In [10]:
import os

def log(func):
    # Ensure log folder exists
    os.makedirs("log", exist_ok=True)    
    
    with open(f"log/{func.__name__}.txt", mode="a") as file_obj:
        file_obj.write(f"[Object defined] {func.__name__}\n")
    
    def wrapper(*args, **kwargs):
        # Log call
        with open(f"log/{func.__name__}.txt", mode="a") as file_obj:
            file_obj.write(
                f"[Object called] {func.__name__} "
                f"with positional arguments {args} "
                f"and keyword arguments {kwargs}\n"
            )
            
        result = func(*args, **kwargs)
        
        # Log completion
        with open(f"log/{func.__name__}.txt", mode="a") as file_obj:
            file_obj.write(f"[Object finished running] {func.__name__}\n")
        ## Use line below to test where log is
        # print(os.path.abspath("log"))
        return result
    
    return wrapper

This example code is looking more complex as we are now having our
decorator write to an external file, but watch what happens when we
apply this to multiple functions:

In [11]:
from datetime import date, timedelta

@log
def list_recipients(names):
    for n in names:
        print(n)


@log
def next_code_club(current_date: date) -> date:
    start_date = date(2025, 5, 1) 

    if current_date <= start_date:
        return start_date
    
    # Total days since start
    days_since_start = (current_date - start_date).days
    
    # cyc
    cycles = days_since_start // 14
    
    # Next event is start_date + (cycles + 1)*14 days
    next_event = start_date + timedelta(days=(cycles + 1) * 14)

    return next_event

print(next_code_club(date(2026,4,29)))
list_recipients(['Michael B. Jordan', 'Adrian Brody', 'Cillian Murphy', 'Brandon Fraser'])
print(next_code_club(date(2023,1,1)))

2026-04-30
Michael B. Jordan
Adrian Brody
Cillian Murphy
Brandon Fraser
2025-05-01

Note how our decorator has created separate files in the log folder for
each of the functions that has been decorated.

Before we move to our third example, it’s worth noting that we can stack
decorators on functions, lets try this with a brief example below:

In [12]:
@func_timer
@log
def slow_function(run_time):
    print("I'll get on that")
    time.sleep(run_time)
    print("I've done that")
    return run_time

print(slow_function(5))

I'll get on that
I've done that
[TIME] wrapper took 5.0016 seconds
5

Note how log has created a log for slow_function, but timer refers to
wrapper. Essentially our timer function is seeing the wrapper from the
log decorator and not the decorated function to enable our decorators to
see the innermost function (in this case slow_function) we can to use
the wraps decorator from the functools library to decorate our wrapper
functions. this lets them inherit properties such as `__name__` from the
function they are wrapping.

If you recall we used the `@wraps` in our timer function so lets see
what happens if we stack them the other way round.

In [13]:
@log
@func_timer
def slow_function_2(run_time):
    print("I'll get on that")
    time.sleep(run_time)
    print("I've done that")
    return run_time

print(slow_function_2(8))

I'll get on that
I've done that
[TIME] slow_function_2 took 8.0013 seconds
8

It is recommended that before using the log function in other work you
add the `@wraps` decorator, we will be using it in the last example as
this is considered good practice. We recommend using it in all
decorators.

### Parameterisation

In analytics work, many reports share the same structure but differ in
*parameters* such as:

- Organisation code  
- Report name  
- Run timestamp  
- Output metadata

A decorator is a clean way to attach this metadata to any reporting
function without repeating boilerplate. To do so we need to add an outer
layer to our decorator factory:

``` python

def report_metadata(org_code: str, report_name: str):
    """
    Decorator that injects metadata into reporting functions.
    """
    def decorator(func):
        @wraps(func)
        def wrapper(*args, **kwargs):
            metadata = {
                "org_code": org_code,
                "report_name": report_name,
                "run_timestamp": datetime.now().isoformat(timespec="seconds")
            }

            print(f"[INFO] Running report '{report_name}' for {org_code}")
            print(f"[INFO] Metadata: {metadata}")

            # Pass metadata into the function
            return func(*args, metadata=metadata, **kwargs)

        return wrapper
    return decorator
```

notice how we pass the metadata in as arguments in the first layer. We
pass these to our inner layer via the decorator function which takes the
function as it’s argument but exists in the same scope as the meta data
meaning the wrapper function can inherit this.

In [14]:
# import pandas as pd

# @report_metadata(org_code="R1F", report_name="Activity Summary")
# def generate_activity_report(df, metadata=None):
#     """
#     Example reporting function that receives metadata from the decorator.
#     """
#     summary = df.describe()
#     return {
#         "metadata": metadata,
#         "summary": summary
#     }


# df = pd.DataFrame({
#     "activity": [10, 12, 9, 14, 11],
#     "cost": [200, 240, 180, 300, 220]
# })

# result = generate_activity_report(df)
# print(result["metadata"])

It is worth noting that for parameterisation to be effective in this
example the function we wrap must be ready to accept the meta data we
are passing it. We can give this a default value of **None** to ensure
default handling.

## Example: Using a Decorator to Parameterise Reporting

In analytics work, many reports share the same structure but differ in
*parameters* such as:

- organisation code  
- report name  
- run timestamp  
- output metadata

A decorator is a clean way to attach this metadata to any reporting
function without repeating boilerplate.

``` python

def report_metadata(org_code: str, report_name: str):
    """
    Decorator that injects metadata into reporting functions.
    """
    def decorator(func):
        @wraps(func)
        def wrapper(*args, **kwargs):
            metadata = {
                "org_code": org_code,
                "report_name": report_name,
                "run_timestamp": datetime.now().isoformat(timespec="seconds")
            }

            print(f"[INFO] Running report '{report_name}' for {org_code}")
            print(f"[INFO] Metadata: {metadata}")

            # Pass metadata into the function
            return func(*args, metadata=metadata, **kwargs)

        return wrapper
    return decorator
```

## Potential Issues

This section summarises five common issues with decorators and offers
some hints to resolve these.

### Decorated Functions Losing Metadata

This is caused by the fact that the wrapper is not inheriting the
decorated functions properties. it is resolved by importing “wraps” from
“functools” and wrapping your wrapper with `@wraps(func)`.

### Decorators Breaking Functions with Arguments

The most likely cause here is that the wrapper cannot handle all the
arguments of thefunction it is wrapping remember to always use “\*args,
\*\*kwargs” in your wrapper and func calls, this will allow your
decorator to handle any number of arguments.

### Decorators With arguments Confuse People (Three‑Layer Pattern)

If you are passing any arguments into the decorated function you may
need to ensure you are following the correct syntax, this is the broad
structure:

``` python

def outer(param):
    def decorator(func):
        def wrapper(*args, **kwargs):
            return func(*args, **kwargs)
        return wrapper
    return decorator
```

### Decorator Order Matters

Ideally decorators should be generalised as possible however you should
check how the decorators you plan to use interact. Some decorators to
not stack on others and you may need to revise the order you stack
decorators if they are causing issues. Remember when stacking 2 or more
decorators, order makes a genuine difference.

### Hidden Side-Effects

decorators can have additional side effects, see this [Educative
site](https://www.educative.io/courses/clean-code-in-python/dealing-with-side-effects-in-decorators?utm_source=copilot.com)
for more information on these. you should always test and debug
functions (including decorators) before decorating them and then test
the effects of adding the decorator(s) you are using.

## Conclusion

This is an incredibly broad subject and there are endless interactions
it is also worth noting that decorators can be applied to certain other
contexts such as decorating methods in classes but these are well
outside the scope of this session and if you wish to understand more we
recommend reviewing some of the following sources:

- [Educative course on
  decorators](https://www.educative.io/courses/clean-code-in-python/what-are-decorators-in-python)
- [Python Decorators - Visually
  Explained](https://www.youtube.com/watch?v=3tyaO-OE0K0)
- [Corey Schaffers tutorial on
  Decorators](https://www.youtube.com/watch?v=FsAPt_9Bf3U)
- [Geeks For Geeks - Decorators in
  Python](https://www.geeksforgeeks.org/python/decorators-in-python/)

And of course you can reach out to any of the Code Club team for support
or drop a call in the [Code Club
Help](https://teams.microsoft.com/l/channel/19%3Aaade66d3b9e542c28224d9c2f43519eb%40thread.tacv2/Code%20Help?groupId=38d3487e-3bc5-4d3c-a710-5a9e895ab03d&tenantId=37c354b2-85b0-47f5-b222-07b48d774ee3)
channel.